In [2]:
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [4]:
sample_text = "Hello, I'm a language model,"

In [6]:
model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")

In [8]:
import tiktoken
encoder = tiktoken.get_encoding('gpt2')

In [9]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")


In [10]:
inputs = tokenizer(sample_text, return_tensors="pt")


In [11]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
outputs = model(**inputs, labels=inputs["input_ids"])


In [5]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")

inputs = tokenizer(sample_text, return_tensors="pt")
outputs = model(**inputs, labels=inputs["input_ids"])
loss = outputs.loss
logits = outputs.logits

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


In [6]:
logits

tensor([[[ -35.2361,  -35.3264,  -38.9752,  ...,  -44.4643,  -43.9973,
           -36.4578],
         [-112.6171, -114.5832, -116.5725,  ..., -119.0127, -118.8060,
          -111.6917],
         [-151.7890, -152.3330, -156.7317,  ..., -162.0787, -155.4330,
          -154.7270],
         ...,
         [-101.2856, -102.6806, -106.1685,  ..., -111.2953, -112.3794,
          -104.9979],
         [-101.5027, -103.5055, -108.4597,  ..., -116.2317, -114.9146,
          -105.6841],
         [-103.7559, -105.5973, -106.9940,  ..., -110.1292, -110.7860,
          -104.5280]]], grad_fn=<UnsafeViewBackward0>)

In [8]:
# Apply softmax to get probabilities
probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)
print(probas.shape)
# Get the idx of the vocab entry with the highest probability value
idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)
# Append sampled index to the running sequence
idx = torch.cat((inputs['input_ids'], idx_next.squeeze(-1)), dim=0)  # (batch, n_tokens+1)

torch.Size([1, 8, 50257])


In [10]:
import tiktoken
encoder = tiktoken.get_encoding('gpt2')

In [11]:
encoder.decode(idx.flatten().tolist())

"Hello, I'm a language model,, I'm sorry little experter not"